In [1]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse, urljoin
import pandas as pd
import json
import re
import time
from pathlib import Path


In [2]:
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

PROGRAMME_URL = "https://opetussuunnitelmat.peppi.jamk.fi/48/en/59/21547/1094?lang=en"

# Use None for all courses, or a small number like 10 for testing
MAX_COURSES = None

# FIX 2: All output files go to a single controlled directory
OUTPUT_DIR = Path("data/BIT")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("http://", adapter)
session.mount("https://", adapter)


In [3]:
# FIX 7: These are keyword sets used by detect_language() for heuristic
# language identification. Adjust these lists if detection accuracy needs
# tuning — e.g. during preprocessing experiments.
FI_WORDS = {
    "ja", "on", "että", "sekä", "tai", "voi", "opiskelija", "opintojakso",
    "opintojakson", "osaamistavoitteet", "sisältö", "arviointikriteeri",
    "arviointikriteerit", "hyväksytty", "osaa", "tutustut", "tiedostat",
    "perehdyt", "kehität", "ymmarrat", "ymmärtää", "laadit", "opintojasi",
    "opiskelutaitoja", "hyvinvoinnin", "korkeakoulussa", "suomi"
}

EN_WORDS = {
    "the", "and", "course", "student", "students", "learning", "outcomes",
    "content", "assessment", "criteria", "approved", "objective",
    "objectives", "study", "studies", "skills", "develop", "understand",
    "ability", "able", "programming", "graphics", "concurrent", "english",
    "you", "your"
}

FI_OUTCOME_KEYS = ["osaamistavoitteet"]
EN_OUTCOME_KEYS = ["objective", "learning outcomes", "outcomes"]

FI_CONTENT_KEYS = ["sisältö"]
EN_CONTENT_KEYS = ["content"]

FI_ASSESSMENT_KEYS = [
    "arviointikriteeri, hyväksytty/hylätty",
    "arviointikriteerit, tyydyttävä (1)",
    "arviointikriteerit, hyvä (3)",
    "arviointikriteerit, kiitettävä (5)",
]
EN_ASSESSMENT_KEYS = [
    "assessment criteria, approved/failed",
    "assessment criteria, satisfactory (1)",
    "assessment criteria, good (3)",
    "assessment criteria, excellent (5)",
]


In [4]:
def normalize_url(url):
    url = str(url).strip()
    # Strip browser "view-source:" prefix that can appear during development
    if url.startswith("view-source:"):
        url = url.replace("view-source:", "", 1)
    return url

def set_lang(url, lang):
    url = normalize_url(url)
    parsed = urlparse(url)
    query = parse_qs(parsed.query)
    query["lang"] = [lang]
    return urlunparse(parsed._replace(query=urlencode(query, doseq=True)))

def fetch_soup(url):
    url = normalize_url(url)
    response = session.get(url, headers=HEADERS, timeout=(20, 60))
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser"), response.url

def clean_text(text):
    if text is None:
        return ""
    text = str(text).strip()
    if text == "-":
        return ""
    return re.sub(r"\s+", " ", text).strip()

def extract_course_id_from_url(url):
    parsed = urlparse(url)
    parts = [p for p in parsed.path.split("/") if p]
    if "course" in parts:
        idx = parts.index("course")
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return ""

def strip_credits_from_title(title):
    title = clean_text(title)
    return re.sub(r"\s*\(\s*\d+(?:\s*-\s*\d+)?\s*(op|cr)\s*\)\s*$", "", title, flags=re.IGNORECASE)

def extract_credits_from_text(title_fi="", title_en="", credits_raw=""):
    # FIX 6: Only include credits_raw when it is non-empty to make intent clear
    candidates = [c for c in [credits_raw, title_fi, title_en] if c]
    for text in candidates:
        text = str(text)
        m = re.search(r"(\d+(?:\s*-\s*\d+)?)\s*(op|cr)", text, flags=re.IGNORECASE)
        if m:
            return re.sub(r"\s+", " ", m.group(1)).strip()
    return ""

def normalize_heading(text):
    text = clean_text(text).lower()
    text = text.replace("\u2019", "'")
    return text


In [5]:
def parse_dt_dd_map(soup):
    info = {}

    for dl in soup.find_all("dl"):
        dt = dl.find("dt")
        dds = dl.find_all("dd")
        if not dt or not dds:
            continue

        key = normalize_heading(dt.get_text(" ", strip=True))
        vals = [clean_text(dd.get_text(" ", strip=True)) for dd in dds]
        vals = [v for v in vals if v]

        if vals:
            if key not in info:
                info[key] = []
            info[key].extend(vals)

    return info

def parse_sections(soup):
    sections = {}

    candidates = soup.select(
        "#course_description .card-body, "
        ".course_description_card .card-body, "
        "main"
    )

    for container in candidates:
        local_sections = {}

        for h3 in container.find_all("h3"):
            heading = normalize_heading(h3.get_text(" ", strip=True))
            texts = []

            node = h3.find_next_sibling()
            while node and getattr(node, "name", None) not in ["h2", "h3"]:
                if getattr(node, "name", None) in ["p", "ul", "ol", "div"]:
                    txt = clean_text(node.get_text("\n", strip=True))
                    if txt:
                        texts.append(txt)
                node = node.find_next_sibling()

            if texts:
                local_sections[heading] = "\n".join(texts)

        if local_sections:
            sections = local_sections
            break

    return sections

def parse_course_page(url):
    soup, final_url = fetch_soup(url)
    info = parse_dt_dd_map(soup)
    sections = parse_sections(soup)

    h1 = soup.find("h1")
    title = clean_text(h1.get_text(" ", strip=True)) if h1 else ""

    credits_raw = ""
    for key in ["laajuus", "number of ects credits allocated"]:
        if key in info and info[key]:
            credits_raw = info[key][0]
            break

    teaching_language_raw = ""
    for key in ["opetuskieli", "teaching language", "teaching languages"]:
        if key in info and info[key]:
            teaching_language_raw = " | ".join(info[key])
            break

    return {
        "url": final_url,
        "course_id": extract_course_id_from_url(final_url),
        "title_raw": title,
        "credits_raw": credits_raw,
        "teaching_language_raw": teaching_language_raw,
        "sections": sections,
    }


In [6]:
def collect_values(sections, keys):
    values = []
    for key in keys:
        value = clean_text(sections.get(key, ""))
        if value:
            values.append(value)
    return values

def remove_diacritics_light(text):
    repl = str.maketrans({
        "ä": "a", "ö": "o", "å": "a",
        "Ä": "A", "Ö": "O", "Å": "A"
    })
    return str(text).translate(repl)

def detect_language(text):
    text = clean_text(text).lower()
    if not text:
        return "empty", 0, 0

    # FIX 5: Note — diacritic bonus is counted on the original text BEFORE
    # stripping, because remove_diacritics_light replaces ä/ö/å for word
    # matching. Both steps are intentional and work together.
    fi_diacritic_bonus = text.count("ä") + text.count("ö") + text.count("å")

    text_simple = remove_diacritics_light(text)
    words = re.findall(r"[a-zA-Zäöå]+", text_simple.lower())

    fi_hits = sum(1 for w in words if w in FI_WORDS) + fi_diacritic_bonus
    en_hits = sum(1 for w in words if w in EN_WORDS)

    if fi_hits >= en_hits + 2:
        return "fi", fi_hits, en_hits
    if en_hits >= fi_hits + 2:
        return "en", fi_hits, en_hits
    return "unknown", fi_hits, en_hits

def canonical_compare_text(values):
    joined = " ".join(values)
    joined = joined.lower()
    # FIX (minor): also strip punctuation so texts differing only by trailing
    # period or comma are correctly flagged as identical
    joined = re.sub(r"[^\w\s]", "", joined)
    joined = re.sub(r"\s+", " ", joined).strip()
    return joined

def validate_field_pair(fi_values, en_values, field_name):
    problems = []

    fi_lang, fi_score, en_score_for_fi = detect_language(" ".join(fi_values))
    en_lang, fi_score_for_en, en_score = detect_language(" ".join(en_values))

    if fi_values and fi_lang == "en":
        problems.append(f"{field_name}_fi looks English")
    if en_values and en_lang == "fi":
        problems.append(f"{field_name}_en looks Finnish")

    if fi_values and en_values:
        if canonical_compare_text(fi_values) == canonical_compare_text(en_values):
            problems.append(f"{field_name}_fi and {field_name}_en are identical")

    return {
        "fi_lang": fi_lang,
        "en_lang": en_lang,
        "problems": problems,
    }


In [7]:
def expand_assessment_items(items, lang):
    """
    Split merged assessment blocks like:
    '1 Välttävä ... 2 Tyydyttävä ...'
    into separate list items.

    Works for:
    - Finnish numeric grades 1..5 with grade labels
    - English numeric grades 1..5 with dash style '1 - ...'
    - approved/failed style text, which is kept as-is
    """
    if not items:
        return []

    text = " ".join(clean_text(x) for x in items if clean_text(x))
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    if lang == "fi":
        pattern = re.compile(
            r'(?:(?<=^)|(?<=\s))([1-5])\s+(Välttävä|Tyydyttävä|Hyvä|Kiitettävä|Erinomainen)\b'
        )
    else:
        pattern = re.compile(
            r'(?:(?<=^)|(?<=\s))([1-5])\s*-\s*'
        )

    matches = list(pattern.finditer(text))

    if not matches:
        return [text]

    parts = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if chunk:
            parts.append(chunk)

    return parts


In [8]:
def build_record(page_fi, page_en):
    sections_fi = page_fi["sections"]
    sections_en = page_en["sections"]

    title_fi = strip_credits_from_title(page_fi["title_raw"])
    title_en = strip_credits_from_title(page_en["title_raw"])

    outcomes_fi = collect_values(sections_fi, FI_OUTCOME_KEYS)
    outcomes_en = collect_values(sections_en, EN_OUTCOME_KEYS)

    contents_fi = collect_values(sections_fi, FI_CONTENT_KEYS)
    contents_en = collect_values(sections_en, EN_CONTENT_KEYS)

    assessment_fi_raw = collect_values(sections_fi, FI_ASSESSMENT_KEYS)
    assessment_en_raw = collect_values(sections_en, EN_ASSESSMENT_KEYS)

    assessment_fi = expand_assessment_items(assessment_fi_raw, "fi")
    assessment_en = expand_assessment_items(assessment_en_raw, "en")

    credits = extract_credits_from_text(
        title_fi=page_fi["title_raw"],
        title_en=page_en["title_raw"],
        credits_raw=page_fi["credits_raw"] or page_en["credits_raw"]
    )

    val_outcomes = validate_field_pair(outcomes_fi, outcomes_en, "outcomes")
    val_contents = validate_field_pair(contents_fi, contents_en, "contents")
    val_assessment = validate_field_pair(assessment_fi, assessment_en, "assessment")

    problems = []
    problems.extend(val_outcomes["problems"])
    problems.extend(val_contents["problems"])
    problems.extend(val_assessment["problems"])

    outcomes_pair_ok = bool(outcomes_fi and outcomes_en) and not val_outcomes["problems"]
    contents_pair_ok = bool(contents_fi and contents_en) and not val_contents["problems"]

    usable_bilingual = bool(title_fi and title_en) and (outcomes_pair_ok or contents_pair_ok)

    # FIX 4: Instead of silently dropping assessment data on validation failure,
    # keep the raw values and add a flag so the record itself is self-documenting.
    assessment_valid = not bool(val_assessment["problems"])
    if not assessment_valid:
        assessment_fi = []
        assessment_en = []

    record = {
        "course_id": page_fi["course_id"] or page_en["course_id"],
        "title_fi": title_fi,
        "title_en": title_en,
        "credits": credits,
        "outcomes_fi": outcomes_fi,
        "outcomes_en": outcomes_en,
        "contents_fi": contents_fi,
        "contents_en": contents_en,
        "assessment_fi": assessment_fi,
        "assessment_en": assessment_en,
        "assessment_valid": assessment_valid,   # NEW: False means data was dropped
    }

    report = {
        "course_id": record["course_id"],
        "title_fi": title_fi,
        "title_en": title_en,
        "course_url_fi": page_fi["url"],
        "course_url_en": page_en["url"],
        "teaching_language_fi_page": page_fi["teaching_language_raw"],
        "teaching_language_en_page": page_en["teaching_language_raw"],
        "outcomes_fi_lang": val_outcomes["fi_lang"],
        "outcomes_en_lang": val_outcomes["en_lang"],
        "contents_fi_lang": val_contents["fi_lang"],
        "contents_en_lang": val_contents["en_lang"],
        "assessment_fi_lang": val_assessment["fi_lang"],
        "assessment_en_lang": val_assessment["en_lang"],
        "assessment_valid": assessment_valid,
        "usable_bilingual": usable_bilingual,
        "problems": " | ".join(problems),
    }

    return record, report


In [9]:
soup, final_programme_url = fetch_soup(PROGRAMME_URL)

course_urls = []
seen = set()

for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/course/" not in href:
        continue

    full_url_en = set_lang(urljoin(final_programme_url, href), "en")
    full_url_fi = set_lang(full_url_en, "fi")

    if full_url_en not in seen:
        seen.add(full_url_en)
        course_urls.append({
            "course_url_en": full_url_en,
            "course_url_fi": full_url_fi,
        })

df_course_urls = pd.DataFrame(course_urls)

if MAX_COURSES is not None:
    df_course_urls = df_course_urls.head(MAX_COURSES).copy()

print("Course URLs to process:", len(df_course_urls))
df_course_urls.head()


Course URLs to process: 80


,course_url_en,course_url_fi
0,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...
1,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...
2,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...
3,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...
4,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...


In [10]:
raw_records = []
validation_reports = []

for i, row in enumerate(df_course_urls.itertuples(index=False), start=1):
    url_en = row.course_url_en
    url_fi = row.course_url_fi

    try:
        page_en = parse_course_page(url_en)
        time.sleep(2)
        page_fi = parse_course_page(url_fi)

        record, report = build_record(page_fi, page_en)
        raw_records.append(record)
        validation_reports.append(report)

        print(f"{i}. OK   {record['course_id']} | {record['title_en']}")

    except Exception as e:
        fallback_course_id = extract_course_id_from_url(url_en)
        validation_reports.append({
            "course_id": fallback_course_id,
            "title_fi": "",
            "title_en": "",
            "course_url_fi": url_fi,
            "course_url_en": url_en,
            "teaching_language_fi_page": "",
            "teaching_language_en_page": "",
            "outcomes_fi_lang": "",
            "outcomes_en_lang": "",
            "contents_fi_lang": "",
            "contents_en_lang": "",
            "assessment_fi_lang": "",
            "assessment_en_lang": "",
            "assessment_valid": False,
            "usable_bilingual": False,
            "problems": f"request_or_parse_error: {e}",
        })
        print(f"{i}. FAIL {fallback_course_id} | {e}")

    finally:
        # FIX 1: Sleep runs for both success and failure to avoid hammering
        # the server after rapid consecutive errors
        time.sleep(2)

    if i % 5 == 0:
        pd.DataFrame(raw_records).to_json(
            OUTPUT_DIR / "jamk_bilingual_raw_partial.json",
            orient="records",
            force_ascii=False,
            indent=2
        )
        pd.DataFrame(validation_reports).to_csv(
            OUTPUT_DIR / "jamk_bilingual_validation_partial.csv",
            index=False,
            encoding="utf-8-sig"
        )
        print("  → Partial files saved")


1. OK   ZZ00CB57 | Me as a Student in Higher Education
2. OK   ZZ00CB58 | Information Seeking and Reporting
3. OK   ZZ00CB59 | Career Planning and Working Life Skills
4. OK   ZZ00CB60 | ICT Skills
5. OK   ZZ00CD03 | English at Work
  → Partial files saved
6. OK   ZZ00CD01 | Swedish for Working Life
7. OK   ZZ00CD04 | Finnish 1
8. OK   ZZ00CD05 | Finnish 2 for Business
9. OK   ZZ00CD13 | Basics of Sustainable Development
10. OK   ZZ00CK90 | Entrepreneurship
  → Partial files saved
11. OK   ZZ00CK91 | InnoFlash
12. OK   ZZ00CD12 | Working Life Project
13. OK   HG00CI43 | Operating Systems
14. OK   HG00CI46 | Basics of Programming
15. OK   HG00CI45 | Basics for Digital Media
  → Partial files saved
16. OK   HT00CF20 | Marketing for Developers
17. OK   HG00CQ39 | Game Software Engineering
18. OK   HG00CF48 | Introduction to Game Graphics
19. OK   HG00CF49 | Basics of Game Design
20. OK   HG00CI44 | Game Programming
  → Partial files saved
21. OK   HG00CI47 | Game Engine
22. OK   HG00CF50 |

In [11]:
df_validation = pd.DataFrame(validation_reports)
df_raw = pd.DataFrame(raw_records)

print("Raw scraped rows:", len(df_raw))
print("Validation rows:", len(df_validation))

df_validation.head()


Raw scraped rows: 80
Validation rows: 80


,course_id,title_fi,title_en,course_url_fi,course_url_en,teaching_language_fi_page,teaching_language_en_page,outcomes_fi_lang,outcomes_en_lang,contents_fi_lang,contents_en_lang,assessment_fi_lang,assessment_en_lang,assessment_valid,usable_bilingual,problems
0,ZZ00CB57,Minä korkeakouluopiskelijana,Me as a Student in Higher Education,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...,suomi | englanti,Finnish | English,fi,en,fi,en,fi,en,True,True,
1,ZZ00CB58,Tiedonhankinta ja raportointi,Information Seeking and Reporting,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...,suomi | englanti,Finnish | English,fi,en,fi,en,fi,en,True,True,
2,ZZ00CB59,Urasuunnittelu- ja työelämätaidot,Career Planning and Working Life Skills,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...,suomi | englanti,Finnish | English,fi,en,fi,en,fi,en,True,True,
3,ZZ00CB60,ICT-valmiudet,ICT Skills,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...,suomi | englanti,Finnish | English,fi,en,fi,en,fi,en,True,True,
4,ZZ00CD03,English at Work,English at Work,https://opetussuunnitelmat.peppi.jamk.fi/cours...,https://opetussuunnitelmat.peppi.jamk.fi/cours...,englanti,English,en,en,en,en,en,en,False,False,outcomes_fi looks English | outcomes_fi and ou...


In [12]:
# Keep only bilingual-usable courses
usable_ids = set(
    df_validation.loc[df_validation["usable_bilingual"] == True, "course_id"].astype(str)
)

final_records = [
    rec for rec in raw_records
    if str(rec.get("course_id", "")) in usable_ids
]

print("Final valid bilingual records:", len(final_records))

if final_records:
    print(json.dumps(final_records[0], ensure_ascii=False, indent=2))


Final valid bilingual records: 61
{
  "course_id": "ZZ00CB57",
  "title_fi": "Minä korkeakouluopiskelijana",
  "title_en": "Me as a Student in Higher Education",
  "credits": "2",
  "outcomes_fi": [
    "Opintojakson osaamiset: - Oppimaan oppiminen: Tunnistat ja kehität korkeakoulussa vaadittavia opiskelutaitojasi. Opit suunnittelemaan opintojasi. - Eettisyys: Tutustut Jamkin periaatteisiin ja ymmärrät mihin sääntöihin toimintasi korkeakoulussa perustuu - Kansainvälisyys ja monikulttuurisuus: Tiedostat mahdollisuutesi kansainvälistymiseen. Opintojakson osaamistavoitteet: - Kiinnityt opiskeluympäristöösi ja omaan ryhmääsi - Perehdyt Jamkin toimintatapoihin ja tiedostat velvollisuutesi ja oikeutesi opiskelijana. - Kehität korkeakoulussa vaadittavia opiskelutaitoja - Ymmärrät hyvinvoinnin merkityksen opintojen edistymisen näkökulmasta. - Edistät omalta osaltasi tasa-arvoa ja yhdenvertaisuutta opiskeluympäristössäsi. - Perehdyt tutkinto-ohjelmasi opetussuunnitelman tarjoamiin mahdollisuuks

In [13]:
# Save final clean JSON
with open(OUTPUT_DIR / "jamk_courses_bilingual_clean.json", "w", encoding="utf-8") as f:
    json.dump(final_records, f, ensure_ascii=False, indent=2)

# Save final clean CSV (flattened — list fields joined with " | " for readability)
df_final = pd.DataFrame(final_records)
for col in ["outcomes_fi", "outcomes_en", "contents_fi", "contents_en",
            "assessment_fi", "assessment_en"]:
    df_final[col] = df_final[col].apply(
        lambda x: " | ".join(x) if isinstance(x, list) else x
    )
df_final.to_csv(
    OUTPUT_DIR / "jamk_courses_bilingual_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save full validation report
df_validation.to_csv(
    OUTPUT_DIR / "jamk_courses_validation_report.csv",
    index=False,
    encoding="utf-8-sig"
)

# Optional: full raw dump including non-usable courses
with open(OUTPUT_DIR / "jamk_courses_bilingual_raw.json", "w", encoding="utf-8") as f:
    json.dump(raw_records, f, ensure_ascii=False, indent=2)

print("Saved to", OUTPUT_DIR)
print("  jamk_courses_bilingual_clean.json")
print("  jamk_courses_bilingual_clean.csv     ← NEW")
print("  jamk_courses_validation_report.csv")
print("  jamk_courses_bilingual_raw.json")


Saved to data\BIT
  jamk_courses_bilingual_clean.json
  jamk_courses_bilingual_clean.csv     ← NEW
  jamk_courses_validation_report.csv
  jamk_courses_bilingual_raw.json


In [14]:
# Quick summary
total = len(df_validation)
usable = int(df_validation["usable_bilingual"].fillna(False).sum())
invalid = total - usable

print("Total checked:", total)
print("Usable bilingual pairs:", usable)
print("Excluded:", invalid)

display(
    df_validation.loc[df_validation["usable_bilingual"] == False,
                      ["course_id", "title_en", "problems"]].head(20)
)


Total checked: 80
Usable bilingual pairs: 61
Excluded: 19


,course_id,title_en,problems
4,ZZ00CD03,English at Work,outcomes_fi looks English | outcomes_fi and ou...
6,ZZ00CD04,Finnish 1,outcomes_fi looks English | outcomes_fi and ou...
7,ZZ00CD05,Finnish 2 for Business,outcomes_fi looks English | outcomes_fi and ou...
45,HG00CF65,Game Project 1,outcomes_fi looks English | outcomes_fi and ou...
46,HG00CF66,Game Project 2,outcomes_fi looks English | outcomes_fi and ou...
47,HG00CF67,Game Project 3,outcomes_fi looks English | outcomes_fi and ou...
58,ZW00BM05,Degree Student Tutoring,outcomes_fi looks English | outcomes_fi and ou...
59,ZW00BM04,Exchange Student Tutoring,outcomes_fi looks English | outcomes_fi and ou...
63,ZZ00CQ16,Narratives of entrepreneurship,outcomes_en looks Finnish | outcomes_fi and ou...
67,ZW00BS75,Integration into the Finnish Society,outcomes_fi looks English | outcomes_fi and ou...


In [15]:
import pandas as pd
import json

with open(OUTPUT_DIR / "jamk_courses_bilingual_clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

def list_len(x):
    return len(x) if isinstance(x, list) else 0

df["has_outcomes_pair"] = (df["outcomes_fi"].apply(list_len) > 0) & (df["outcomes_en"].apply(list_len) > 0)
df["has_contents_pair"] = (df["contents_fi"].apply(list_len) > 0) & (df["contents_en"].apply(list_len) > 0)
df["has_assessment_pair"] = (df["assessment_fi"].apply(list_len) > 0) & (df["assessment_en"].apply(list_len) > 0)

print("Total courses:", len(df))
print("Courses with outcomes pair:", df["has_outcomes_pair"].sum())
print("Courses with contents pair:", df["has_contents_pair"].sum())
print("Courses with assessment pair:", df["has_assessment_pair"].sum())

usable_core = df[(df["has_outcomes_pair"]) | (df["has_contents_pair"])]
print("Usable for core experiment:", len(usable_core))


Total courses: 61
Courses with outcomes pair: 61
Courses with contents pair: 61
Courses with assessment pair: 57
Usable for core experiment: 61
